# Ollama + T4 Vision → Adobe Stock Metadata

Google Colab T4 par Ollama vision model chalao. Colab ke liye required `zstd` dependency automatically install hoti hai., images ko visually analyze karo, aur exact Adobe Stock CSV generate karo.

**Output:** `AdobeStock_Metadata.csv` + `AdobeStock_Metadata.json`

CSV: `Filename,Title,Keywords,Category,Releases`


In [ ]:
# 1) T4/GPU check
!nvidia-smi


In [ ]:
# 2) Install Ollama dependencies + Ollama
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version


In [ ]:
# 3) Start Ollama service
import subprocess, time, shutil

if shutil.which("ollama") is None:
    raise RuntimeError("Ollama is not installed. Re-run Cell 2 first.")

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
!ollama list


In [ ]:
# 4) Pull vision model
MODEL = "llama3.2-vision:11b"
!ollama pull llama3.2-vision:11b
!ollama list


In [ ]:
# 5) Upload images
from google.colab import files
from pathlib import Path

IMAGE_DIR = Path("/content/images")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    (IMAGE_DIR / Path(name).name).write_bytes(data)

print(f"Uploaded: {len(uploaded)}")


## 6) Metadata rules

AI must actually inspect each image. It must not infer content from filenames. Each image gets its own title, keywords, Adobe numeric category and releases field.


In [ ]:
# 7) One-image vision test
import json, subprocess
from pathlib import Path

PROMPT = r"""
Analyze the supplied image visually for Adobe Stock metadata.

Return ONLY valid JSON:
{
  "title": "...",
  "keywords": ["..."],
  "category": 0,
  "releases": ""
}

Rules:
- Analyze the actual visible image.
- Never infer content from filename.
- Title <= 200 characters, accurate and natural.
- Maximum 49 relevant keywords, strongest first.
- No duplicate or invented keywords.
- Category must be an Adobe Stock numeric category code.
- Releases must be blank unless actual release information is supplied.
- No model/provider/API information.
- JSON only.
"""

def analyze(image_path):
    r = subprocess.run(
        ["ollama", "run", MODEL, PROMPT, str(image_path)],
        capture_output=True, text=True, timeout=180
    )
    if r.returncode:
        raise RuntimeError(r.stderr.strip() or "Ollama failed")
    return r.stdout.strip()

images = sorted([p for p in IMAGE_DIR.iterdir()
                 if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}])
if not images:
    raise RuntimeError("No images found.")
print("Testing:", images[0].name)
raw = analyze(images[0])
print(raw[:4000])


In [ ]:
# 8) Validate one-image result
def extract_json(text):
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        a, b = text.find("{"), text.rfind("}")
        if a >= 0 and b > a:
            return json.loads(text[a:b+1])
        raise

def validate(meta):
    required = {"title","keywords","category","releases"}
    missing = required - set(meta)
    if missing:
        raise ValueError(f"Missing: {missing}")
    if len(str(meta["title"])) > 200:
        raise ValueError("Title > 200 characters")
    if not isinstance(meta["keywords"], list) or len(meta["keywords"]) > 49:
        raise ValueError("Keywords invalid or > 49")
    kws = [str(x).strip().lower() for x in meta["keywords"]]
    if len(kws) != len(set(kws)):
        raise ValueError("Duplicate keywords")
    if not isinstance(meta["category"], int):
        raise ValueError("Category must be numeric")
    return True

test_meta = extract_json(raw)
validate(test_meta)
print("SINGLE IMAGE TEST: PASS")
print(json.dumps(test_meta, indent=2, ensure_ascii=False))


In [ ]:
# 9) Process ALL images sequentially
import csv, time, traceback

OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / "AdobeStock_Metadata.csv"
JSON_PATH = OUTPUT_DIR / "AdobeStock_Metadata.json"
ERROR_PATH = OUTPUT_DIR / "AdobeStock_Metadata_Errors.json"

records, errors = [], []

for i, image_path in enumerate(images, 1):
    print(f"[{i}/{len(images)}] {image_path.name}")
    try:
        meta = extract_json(analyze(image_path))
        validate(meta)
        records.append({
            "Filename": image_path.name,
            "Title": str(meta["title"]).strip(),
            "Keywords": ", ".join(str(k).strip() for k in meta["keywords"]),
            "Category": int(meta["category"]),
            "Releases": str(meta.get("releases", "") or "").strip()
        })
        print("  PASS")
    except Exception as e:
        errors.append({"Filename": image_path.name, "error": str(e)})
        print("  FAILED:", e)
    time.sleep(0.5)

print(f"Successful: {len(records)} | Failed: {len(errors)}")


In [ ]:
# 10) Export exact Adobe Stock CSV + master JSON
headers = ["Filename","Title","Keywords","Category","Releases"]

with CSV_PATH.open("w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=headers)
    w.writeheader()
    w.writerows(records)

JSON_PATH.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")

if errors:
    ERROR_PATH.write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding="utf-8")

print(CSV_PATH)
print(JSON_PATH)


In [ ]:
# 11) Final validation
import pandas as pd

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig", dtype=str).fillna("")
assert list(df.columns) == ["Filename","Title","Keywords","Category","Releases"]
assert len(df) == len(records)

actual_names = {p.name for p in images}
for _, row in df.iterrows():
    assert row.Filename in actual_names
    assert len(row.Title) <= 200
    kws = [x.strip() for x in row.Keywords.split(",") if x.strip()]
    assert len(kws) <= 49
    assert len({x.lower() for x in kws}) == len(kws)
    assert row.Category.isdigit()

print("FINAL VALIDATION: PASS")
print("Images:", len(images))
print("CSV rows:", len(df))
print("Columns:", list(df.columns))


In [ ]:
# 12) Download outputs
from google.colab import files
files.download(str(CSV_PATH))
files.download(str(JSON_PATH))
if ERROR_PATH.exists():
    files.download(str(ERROR_PATH))
